# Data cleaning

Builds the table for the Covid enrollment map from `dataset/ipeds_admissions_panel.csv` and saves it to `clean/map_covid.csv`.
The map itself is drawn by `covid_map.py`. Figure specs are in `CLAUDE.md`.
Run the cells top to bottom.

In [ ]:
from pathlib import Path
import pandas as pd

Path("clean").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

## 1. Load
Only the columns we need.

In [ ]:
cols = ["unitid", "cohort_year", "state", "applicants", "admitted", "enrolled"]
panel = pd.read_csv("dataset/ipeds_admissions_panel.csv", usecols=cols)
panel.shape

## 2. Look before changing anything

In [ ]:
assert not panel.duplicated(["unitid", "cohort_year"]).any()
panel.groupby("cohort_year")["unitid"].nunique()

In [ ]:
panel[["applicants", "admitted", "enrolled"]].isna().mean()

## 3. States
Keep the 50 states and DC (drops territories).

In [ ]:
states = set("""
AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT
NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY
""".split())
panel = panel[panel["state"].isin(states)]
panel["state"].nunique()

## 4. Bad rows
Drop impossible rows. Blanks stay blank, never filled with zero.

In [ ]:
bad = ((panel.admitted > panel.applicants) | (panel.enrolled > panel.admitted)
       | (panel.applicants == 0) | (panel.admitted == 0))
print(bad.sum(), "rows dropped")
panel = panel[~bad].copy()

## 5. Covid table

Percent change in first-year enrollment by state, fall 2019 to fall 2020.
This only needs schools that report in both 2019 and 2020.

In [ ]:
state_of = panel.drop_duplicates("unitid").set_index("unitid")["state"]
w = panel.pivot_table(index="unitid", columns="cohort_year", values="enrolled")
w = w[[2019, 2020]].dropna()
w["state"] = state_of.reindex(w.index)
g = w.groupby("state")[[2019, 2020]].sum()
map_covid = pd.DataFrame({
    "2019-2020": (g[2020] / g[2019] - 1) * 100,
    "n_schools": w.groupby("state").size(),
})
print(len(w), "schools,", len(map_covid), "states")
map_covid.sort_values("2019-2020").head()

In [ ]:
print(map_covid["n_schools"].describe())
map_covid.to_csv("clean/map_covid.csv")